## Exercise 2

In [1]:
import pandas as pd
import numpy as np
import cvxpy as cp

### Point 1)

#### (a) Compute the covariance matrix Σ.

In [2]:
sigma = np.array([0.20, 0.22, 0.25, 0.18, 0.45, 0.80, 0.35, 0.405])

# Correlation Matrix
C = np.array([                                                   
    [1.0, 0.5, 0.3, 0.6, 0.4, 0.3, 0.4, 0.3],
    [0.5, 1.0, 0.3, 0.6, 0.3, 0.2, 0.6, 0.3],
    [0.3, 0.3, 1.0, 0.6, 0.5, 0.4, 0.5, 0.5],
    [0.6, 0.6, 0.6, 1.0, 0.3, 0.7, 0.6, 0.3],
    [0.4, 0.3, 0.5, 0.3, 1.0, 0.5, 0.5, 0.3],
    [0.3, 0.2, 0.4, 0.7, 0.5, 1.0, 0.6, 0.3],
    [0.4, 0.6, 0.5, 0.6, 0.5, 0.6, 1.0, 0.6],
    [0.3, 0.3, 0.5, 0.3, 0.3, 0.3, 0.6, 1.0]
])

# Compute Covariance Matrix 
Sigma = np.outer (sigma, sigma) * C

Sigma_df = pd.DataFrame(Sigma, 
                        columns=[f"Stock {i}" for i in range(1,9)],
                        index=[f"Stock {i}" for i in range(1,9)])

display(Sigma_df.style.format("{:.2%}"))

,Stock 1,Stock 2,Stock 3,Stock 4,Stock 5,Stock 6,Stock 7,Stock 8
Stock 1,4.00%,2.20%,1.50%,2.16%,3.60%,4.80%,2.80%,2.43%
Stock 2,2.20%,4.84%,1.65%,2.38%,2.97%,3.52%,4.62%,2.67%
Stock 3,1.50%,1.65%,6.25%,2.70%,5.62%,8.00%,4.38%,5.06%
Stock 4,2.16%,2.38%,2.70%,3.24%,2.43%,10.08%,3.78%,2.19%
Stock 5,3.60%,2.97%,5.62%,2.43%,20.25%,18.00%,7.88%,5.47%
Stock 6,4.80%,3.52%,8.00%,10.08%,18.00%,64.00%,16.80%,9.72%
Stock 7,2.80%,4.62%,4.38%,3.78%,7.88%,16.80%,12.25%,8.50%
Stock 8,2.43%,2.67%,5.06%,2.19%,5.47%,9.72%,8.50%,16.40%


#### (b) Compute the expected return µ(b), the volatility σ(b) and the Sharpe ratio SR(b | r) of the benchmark.

In [3]:
r = 0.01

b = np.array([0.095, 0.155, 0.055, 0.085, 0.10, 0.25, 0.17, 0.09])
mu = np.array([0.05, 0.055, 0.06, 0.04, 0.07, 0.10, 0.0875, 0.0625])

# expected return
mu_b = mu @ b

# volatility
sigma_b = np.sqrt(b @ Sigma @ b)

# Sharpe ratio
SR_b = (mu_b - r) / sigma_b

print(f"Expected return μ(b): {mu_b:.2%}")
print(f"Volatility σ(b): {sigma_b:.2%}")
print(f"Sharpe ratio SR(b|r): {SR_b:.4f}")

Expected return μ(b): 7.25%
Volatility σ(b): 32.63%
Sharpe ratio SR(b|r): 0.1915


#### (c) We assume that the benchmark b is the optimal market portfolio. Compute the implied expected returns

In [4]:
# Implied expected return of each asset
mu_implied_i = r + SR_b * (Sigma @ b) / sigma_b

# Implied expected return of the portfolio
mu_implied_p = b @ mu_implied_i

df_assets = pd.DataFrame({
    "Stock": range(1, 9),
    "Implied Return": mu_implied_i
})

df_assets["Implied Return"] = df_assets["Implied Return"].map(lambda x: f"{x:.2%}")

display(df_assets.style.hide(axis="index"))

df_portfolio = pd.DataFrame({
    "Portfolio": ["b"],
    "Implied Return": [mu_implied_p]
})

df_portfolio["Implied Return"] = df_portfolio["Implied Return"].map(lambda x: f"{x:.2%}")

display(df_portfolio.style.hide(axis="index"))

Stock,Implied Return
1,2.90%
2,3.03%
3,3.78%
4,3.70%
5,6.68%
6,14.99%
7,6.50%
8,5.11%


Portfolio,Implied Return
b,7.25%


#### (d) Compute the beta βi(b) of each stock with respect to the benchmark b. Deduce the CAPM expected returns:

In [5]:
# compute covariance between each asset and the benchmark portfolio
cov_i_b = Sigma @ b

# compute benchmark variance
var_b = b @ Sigma @ b

# compute CAPM beta of each asset relative to the benchmark
beta = cov_i_b / var_b

# compute CAPM-implied expected returns
mu_capm = r + beta * (mu_b - r)

df = pd.DataFrame({
    "Stock": range(1, len(b)+1),
    "beta_i(b)": beta,     
    "CAPM E[Ri]": mu_capm  
})

df["beta_i(b)"] = df["beta_i(b)"].map(lambda x: f"{x:.4f}")

df["CAPM E[Ri]"] = df["CAPM E[Ri]"].map(lambda x: f"{x:.2%}")

df.style.hide(axis="index")

Stock,beta_i(b),CAPM E[Ri]
1,0.3045,2.90%
2,0.3246,3.03%
3,0.4447,3.78%
4,0.4321,3.70%
5,0.9088,6.68%
6,2.2386,14.99%
7,0.8811,6.50%
8,0.6584,5.11%


In [6]:
print("\nComparison between results of 1(c) and 1(d)\n")

print(f"{'Asset':<6}{'mu_tilde^(b)':>18}{'mu_tilde^(capm)':>20}{'Difference':>15}")
print("-"*60)

for i in range(len(b)):
    diff = mu_capm[i] - mu_implied_i[i]
    print(f"{i+1:<6}{mu_implied_i[i]:>18.2%}{mu_capm[i]:>20.2%}{diff:>15.2%}")

print("-"*60)
print(f"Risk-free rate r = {r:.0%}")


Comparison between results of 1(c) and 1(d)

Asset       mu_tilde^(b)     mu_tilde^(capm)     Difference
------------------------------------------------------------
1                  2.90%               2.90%          0.00%
2                  3.03%               3.03%          0.00%
3                  3.78%               3.78%         -0.00%
4                  3.70%               3.70%         -0.00%
5                  6.68%               6.68%         -0.00%
6                 14.99%              14.99%          0.00%
7                  6.50%               6.50%          0.00%
8                  5.11%               5.11%          0.00%
------------------------------------------------------------
Risk-free rate r = 1%


#### (e) Deduce the alpha of each stock:


In [7]:
# Alpha of each stock
alpha = mu - mu_implied_i

# Create table
df_alpha = pd.DataFrame({
    "Stock": range(1, len(alpha)+1),
    "Alpha": alpha
})

df_alpha["Alpha"] = df_alpha["Alpha"].map(lambda x: f"{x:.2%}")

df_alpha.style.hide(axis="index")

Stock,Alpha
1,2.10%
2,2.47%
3,2.22%
4,0.30%
5,0.32%
6,-4.99%
7,2.25%
8,1.14%


#### (f) Compute CI (b), CM(b), GI (b), and the ESG score S(b).


In [8]:
# define ESG indicators for each asset
CI = np.array([80, 200, 390, 800, 60, 120, 135, 580])                    # Carbon Intensity
CM = np.array([-0.05, -0.075, -0.015, -0.02, 0.08, -0.04, -0.07, 0.02])  # Carbon Momentum
GI = np.array([0.05, 0.805, 0.15, 0.0, 0.02, 0.0, 0.60, 0.20])           # Green Intensity
S = np.array([-2.0, 2.5, 1.5, 2.0, -1.0, -0.5, -0.5, 0.5])               # ESG score

# compute ESG metrics of the benchmark portfolio b
CI_b = b @ CI    # weighted average carbon intensity
CM_b = b @ CM    # weighted average carbon momentum
GI_b = b @ GI    # weighted average green intensity
S_b  = b @ S     # weighted average ESG score

# print the ESG metrics of the benchmark portfolio
print("--- ESG metrics ----")
print(f"CI(b): {CI_b:.4f}")
print(f"CM(b): {CM_b:.2%}")
print(f"GI(b): {GI_b:.2%}")
print(f"S(b): {S_b:.4f}")

--- ESG metrics ----
CI(b): 239.2000
CM(b): -3.10%
GI(b): 25.98%
S(b): 0.1850


#### (g) Compute the carbon intensity, the carbon momentum, the green intensity and the ESG score for each sector

In [9]:
# define the sector of each stock
sector = np.array(["Sector1","Sector2","Sector1","Sector1","Sector2","Sector2","Sector2","Sector1"])

# function to compute ESG metrics for a given sector within the benchmark portfolio
def sector_metrics(sector_name):

    # select assets belonging to the chosen sector
    m = (sector == sector_name)

    # extract benchmark weights of assets in that sector
    w = b[m]

    # normalize weights so that they sum to 1 inside the sector
    w = w / w.sum()

    # compute sector-level ESG metrics using the normalized weights
    return {
        "Sector": sector_name,
        "CI": w @ CI[m],   # sector carbon intensity
        "CM": w @ CM[m],   # sector carbon momentum
        "GI": w @ GI[m],   # sector green intensity
        "S":  w @ S[m],    # sector ESG score
    }

# compute ESG metrics for each sector
res = pd.DataFrame([
    sector_metrics("Sector1"),
    sector_metrics("Sector2")
])

res["CI"] = res["CI"].map(lambda x: f"{x:.2f}")
res["CM"] = res["CM"].map(lambda x: f"{x:.2%}")  
res["GI"] = res["GI"].map(lambda x: f"{x:.2%}")
res["S"]  = res["S"].map(lambda x: f"{x:.3f}")

res.style.hide(axis="index")

Sector,CI,CM,GI,S
Sector1,459.23,-1.68%,9.54%,0.331
Sector2,133.26,-3.78%,33.89%,0.115


### Point 2)

In [10]:
def solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector):

    # solve the ESG-constrained quadratic optimization problem using OSQP
    prob.solve(
        solver=cp.OSQP,
        eps_abs=1e-12,
        eps_rel=1e-12,
        max_iter=200000,
        polish=True,
        verbose=False,
        warm_start=True
    )

    # check if the solver returned a valid solution
    if w.value is None:
        raise RuntimeError(f"Solver failed. Status: {prob.status}")

    # extract optimal portfolio weights and remove very small numerical values
    w_star = np.array(w.value).reshape(-1)
    w_star[np.abs(w_star) < 1e-10] = 0.0

    # compute ESG metrics of the benchmark portfolio
    CI_b = float(CI @ b)   # carbon intensity of benchmark
    CM_b = float(CM @ b)   # carbon momentum of benchmark
    GI_b = float(GI @ b)   # green intensity of benchmark
    S_b  = float(S @ b)    # ESG score of benchmark

    # compute ESG metrics of the optimized portfolio
    CI_w = float(CI @ w_star)   # portfolio carbon intensity
    CM_w = float(CM @ w_star)   # portfolio carbon momentum
    GI_w = float(GI @ w_star)   # portfolio green intensity
    S_w  = float(S @ w_star)    # portfolio ESG score

    # compute tracking error variance and volatility relative to the benchmark
    te_var = float((w_star - b).T @ Sigma @ (w_star - b))
    te_vol = float(np.sqrt(max(te_var, 0.0)))

    # compute sector allocations for both portfolio and benchmark
    sector_alloc = {}
    for sec in np.unique(sector):
        sector_alloc[f"w({sec})"] = float(w_star[sector == sec].sum())  # portfolio sector weight
        sector_alloc[f"b({sec})"] = float(b[sector == sec].sum())       # benchmark sector weight

    # return all results in a dictionary
    return {
        "w_star": w_star,
        "tracking_error_vol": te_vol,
        "tracking_error_var": te_var,
        "CI_benchmark": CI_b,
        "CI_portfolio": CI_w,
        "CM_benchmark": CM_b,
        "CM_portfolio": CM_w,
        "GI_benchmark": GI_b,
        "GI_portfolio": GI_w,
        "S_benchmark": S_b,
        "S_portfolio": S_w,
        "status": prob.status,
        **sector_alloc
    }

#### (a) The investor begins by adding a decarbonization constraint: CI(w) ≤ (1−30%)CI(b) Give the QP problem and find the optimal solution.

In [11]:
# number of assets in the benchmark
n = len(b)

# compute benchmark carbon intensity
CI_b = float(CI @ b)

# set the carbon intensity target 
CI_target = 0.70 * CI_b

# define matrices for the tracking-error quadratic program
Q = Sigma          
R = Sigma @ b      

# define portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# portfolio constraints: fully invested, long-only, and carbon intensity constraint
constraints = [
    cp.sum(w) == 1,     # portfolio weights sum to 1
    w >= 0,             # long-only constraint
    CI @ w <= CI_target # carbon intensity must be below the target
]

# create the ESG-constrained optimization problem
prob = cp.Problem(objective, constraints)

# solve the ESG optimization and compute portfolio metrics
result_2a = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store the carbon intensity target
result_2a["CI_target"] = CI_target

# compute effective variation of carbon intensity relative to the benchmark
result_2a["effective_variation"] = (result_2a["CI_portfolio"] - result_2a["CI_benchmark"]) / result_2a["CI_benchmark"]

# compute effective variation of green intensity relative to the benchmark
result_2a["effective_GI_variation"] = ( result_2a["GI_portfolio"] - result_2a["GI_benchmark"]) / result_2a["GI_benchmark"]

# compute effective variation of carbon momentum relative to the benchmark
result_2a["effective_CM_variation"] = ( result_2a["CM_portfolio"] - result_2a["CM_benchmark"]) / result_2a["CM_benchmark"]

# compute effective variation of ESG score relative to the benchmark
result_2a["effective_S_variation"] = (result_2a["S_portfolio"] - result_2a["S_benchmark"]) / result_2a["S_benchmark"]

In [12]:
w_star = result_2a["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_2a['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_2a['tracking_error_var']:.2%}")

print(f"\nCI(b): {result_2a['CI_benchmark']:.4f}")
print(f"CI(w*): {result_2a['CI_portfolio']:.4f}")
print(f"CI target: {result_2a['CI_target']:.4f}")
print(f"Carbon Intensity Variation: {result_2a['effective_variation']:.2%}")

print(f"\nCM(b): {result_2a['CM_benchmark']:.2%}")
print(f"CM(w*): {result_2a['CM_portfolio']:.2%}")
print(f"Carbon Momentum Variation: {result_2a['effective_CM_variation']:.2%}")

print(f"\nGI(b): {result_2a['GI_benchmark']:.2%}")
print(f"GI(w*): {result_2a['GI_portfolio']:.2%}")
print(f"Green Intensity Variation: {result_2a['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_2a['S_benchmark']:.4f}")
print(f"S(w*): {result_2a['S_portfolio']:.4f}")
print(f"ESG Score Variation: {result_2a['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_2a[f'w({sec})']:.2%} | b = {result_2a[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 16.62%
Stock 2: 16.28%
Stock 3: 6.45%
Stock 4: 0.00%
Stock 5: 9.21%
Stock 6: 25.77%
Stock 7: 19.98%
Stock 8: 5.69%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 1.33%
Tracking error var: 0.02%

CI(b): 239.2000
CI(w*): 167.4400
CI target: 167.4400
Carbon Intensity Variation: -30.00%

CM(b): -3.10%
CM(w*): -3.73%
Carbon Momentum Variation: 20.26%

GI(b): 25.98%
GI(w*): 28.22%
Green Intensity Variation: 8.62%

S(b): 0.1850
S(w*): -0.1209
ESG Score Variation: -165.37%

--- Sector allocation ---
Sector1: w* = 28.76% | b = 32.50%
Sector2: w* = 71.24% | b = 67.50%


#### (b) The investor adds a green intensity constraint: GI(w) ≥(1+50%)GI(b) What does this constraint mean? Give the QP problem and find the optimal solution.

In [13]:
# number of assets in the benchmark
n = len(b)

# compute benchmark carbon intensity and green intensity
CI_b = float(CI @ b)
GI_b = float(GI @ b)

# define target ESG levels relative to the benchmark
CI_target = 0.70 * CI_b   # 30% reduction in carbon intensity
GI_target = 1.50 * GI_b   # 50% increase in green intensity

# define matrices for the tracking-error optimization problem
Q = Sigma          
R = Sigma @ b      

# define portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to the benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# fully invested, long-only, lower carbon intensity, higher green intensity
constraints = [
    cp.sum(w) == 1,      # portfolio weights must sum to 1
    w >= 0,              # long-only constraint
    CI @ w <= CI_target, # carbon intensity target
    GI @ w >= GI_target  # green intensity target
]

# create the ESG-constrained quadratic optimization problem
prob = cp.Problem(objective, constraints)

# solve the optimization problem and compute portfolio metrics
result_2b = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store target values in the result dictionary
result_2b["CI_target"] = CI_target
result_2b["GI_target"] = GI_target

# compute effective variation of carbon intensity relative to the benchmark
result_2b["effective_CI_variation"] = ( result_2b["CI_portfolio"] - result_2b["CI_benchmark"]) / result_2b["CI_benchmark"]

# compute effective variation of green intensity relative to the benchmark
result_2b["effective_GI_variation"] = (result_2b["GI_portfolio"] - result_2b["GI_benchmark"]) / result_2b["GI_benchmark"]

# compute effective variation of carbon momentum relative to the benchmark
result_2b["effective_CM_variation"] = (result_2b["CM_portfolio"] - result_2b["CM_benchmark"]) / result_2b["CM_benchmark"]

# compute effective variation of ESG score relative to the benchmark
result_2b["effective_S_variation"] = (result_2b["S_portfolio"] - result_2b["S_benchmark"]) / result_2b["S_benchmark"]

In [14]:
w_star = result_2b["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_2b['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_2b['tracking_error_var']:.2%}")

print(f"\nCI(b): {result_2b['CI_benchmark']:.4f}")
print(f"CI(w*): {result_2b['CI_portfolio']:.4f}")
print(f"CI target: {result_2b['CI_target']:.4f}")
print(f"Carbon Intensity Variation: {result_2b['effective_CI_variation']:.2%}")

print(f"\nCM(b): {result_2b['CM_benchmark']:.2%}")
print(f"CM(w*): {result_2b['CM_portfolio']:.2%}")
print(f"Carbon Momentum Variation: {result_2b['effective_CM_variation']:.2%}")


print(f"\nGI(b): {result_2b['GI_benchmark']:.2%}")
print(f"GI(w*): {result_2b['GI_portfolio']:.2%}")
print(f"GI target: {result_2b['GI_target']:.2%}")
print(f"Green Intensity Variation: {result_2b['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_2b['S_benchmark']:.4f}")
print(f"S(w*): {result_2b['S_portfolio']:.4f}")
print(f"ESG Score Variation: {result_2b['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_2b[f'w({sec})']:.2%} | b = {result_2b[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 6.38%
Stock 2: 29.82%
Stock 3: 3.20%
Stock 4: 0.00%
Stock 5: 8.75%
Stock 6: 25.72%
Stock 7: 21.90%
Stock 8: 4.23%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 3.21%
Tracking error var: 0.10%

CI(b): 239.2000
CI(w*): 167.4400
CI target: 167.4400
Carbon Intensity Variation: -30.00%

CM(b): -3.10%
CM(w*): -4.38%
Carbon Momentum Variation: 41.33%

GI(b): 25.98%
GI(w*): 38.97%
GI target: 38.97%
Green Intensity Variation: 50.00%

S(b): 0.1850
S(w*): 0.3614
ESG Score Variation: 95.35%

--- Sector allocation ---
Sector1: w* = 13.82% | b = 32.50%
Sector2: w* = 86.18% | b = 67.50%


#### (c) The investor adds a third constraint: CM(w)≤(1+50%)CM(b). What does this constraint mean? Give the QP problem and find the optimal solution

In [15]:
# number of assets in the benchmark
n = len(b)

# compute benchmark ESG metrics
CI_b = float(CI @ b)   # benchmark carbon intensity
GI_b = float(GI @ b)   # benchmark green intensity
CM_b = float(CM @ b)   # benchmark carbon momentum

# define target ESG levels relative to the benchmark
CI_target = 0.70 * CI_b   # 30% reduction in carbon intensity
GI_target = 1.50 * GI_b   # 50% increase in green intensity
CM_target = 1.50 * CM_b   # target level for carbon momentum

# define matrices for the tracking-error optimization problem
Q = Sigma          
R = Sigma @ b     

# define portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to the benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# fully invested, long-only, lower carbon intensity, higher green intensity, and carbon momentum constraint
constraints = [
    cp.sum(w) == 1,      # portfolio weights must sum to 1
    w >= 0,              # long-only constraint
    CI @ w <= CI_target, # carbon intensity target
    GI @ w >= GI_target, # green intensity target
    CM @ w <= CM_target  # carbon momentum target
]

# create the ESG-constrained quadratic optimization problem
prob = cp.Problem(objective, constraints)

# solve the optimization problem and compute portfolio metrics
result_2c = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store target values in the result dictionary
result_2c["CI_target"] = CI_target
result_2c["GI_target"] = GI_target
result_2c["CM_target"] = CM_target

# compute effective variation of carbon intensity relative to the benchmark
result_2c["effective_CI_variation"] = (result_2c["CI_portfolio"] - result_2c["CI_benchmark"]) / result_2c["CI_benchmark"]

# compute effective variation of green intensity relative to the benchmark
result_2c["effective_GI_variation"] = (result_2c["GI_portfolio"] - result_2c["GI_benchmark"]) / result_2c["GI_benchmark"]

# compute effective variation of carbon momentum relative to the benchmark
result_2c["effective_CM_variation"] = (result_2c["CM_portfolio"] - result_2c["CM_benchmark"]) / result_2c["CM_benchmark"]

# compute effective variation of ESG score relative to the benchmark
result_2c["effective_S_variation"] = (result_2c["S_portfolio"] - result_2c["S_benchmark"]) / result_2c["S_benchmark"]

In [16]:
w_star = result_2c["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_2c['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_2c['tracking_error_var']:.2%}")

print(f"\nCI(b): {result_2c['CI_benchmark']:.4f}")
print(f"CI(w*): {result_2c['CI_portfolio']:.4f}")
print(f"CI target: {result_2c['CI_target']:.4f}")
print(f"Carbon Intensity Variation: {result_2c['effective_CI_variation']:.2%}")

print(f"\nCM(b): {result_2c['CM_benchmark']:.2%}")
print(f"CM(w*): {result_2c['CM_portfolio']:.2%}")
print(f"CM target: {result_2c['CM_target']:.2%}")
print(f"Carbon Momentum Variation: {result_2c['effective_CM_variation']:.2%}")

print(f"\nGI(b): {result_2c['GI_benchmark']:.2%}")
print(f"GI(w*): {result_2c['GI_portfolio']:.2%}")
print(f"GI target: {result_2c['GI_target']:.2%}")
print(f"Green Intensity Variation: {result_2c['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_2c['S_benchmark']:.4f}")
print(f"S(w*): {result_2c['S_portfolio']:.4f}")
print(f"ESG Score Variation: {result_2c['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_2c[f'w({sec})']:.2%} | b = {result_2c[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 7.63%
Stock 2: 29.08%
Stock 3: 4.18%
Stock 4: 0.00%
Stock 5: 6.74%
Stock 6: 25.95%
Stock 7: 22.82%
Stock 8: 3.60%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 3.29%
Tracking error var: 0.11%

CI(b): 239.2000
CI(w*): 167.4400
CI target: 167.4400
Carbon Intensity Variation: -30.00%

CM(b): -3.10%
CM(w*): -4.65%
CM target: -4.65%
Carbon Momentum Variation: 50.00%

GI(b): 25.98%
GI(w*): 38.97%
GI target: 38.97%
Green Intensity Variation: 50.00%

S(b): 0.1850
S(w*): 0.3440
ESG Score Variation: 85.94%

--- Sector allocation ---
Sector1: w* = 15.41% | b = 32.50%
Sector2: w* = 84.59% | b = 67.50%


#### (d) The investor adds a fourth constraint: S(w) ≥ S(b)+0.5 What does this constraint mean? Give the QP problem and find the optimal solution

In [17]:
# number of assets in the benchmark
n = len(b)

# compute benchmark ESG metrics
CI_b = float(CI @ b)   # benchmark carbon intensity
GI_b = float(GI @ b)   # benchmark green intensity
S_b  = float(S @ b)    # benchmark ESG score

# define target ESG levels relative to the benchmark
CI_target = 0.70 * CI_b     # 30% reduction in carbon intensity
GI_target = 1.50 * GI_b     # 50% increase in green intensity
CM_target = 1.50 * CM_b     # carbon momentum target (relative to benchmark)
S_target  = S_b + 0.50      # ESG score improvement target

# define matrices for the tracking-error optimization problem
Q = Sigma          
R = Sigma @ b      

# define portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to the benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# fully invested, long-only, and ESG/climate constraints
constraints = [
    cp.sum(w) == 1,      # portfolio weights must sum to 1
    w >= 0,              # long-only constraint
    CI @ w <= CI_target, # carbon intensity constraint
    GI @ w >= GI_target, # green intensity constraint
    CM @ w <= CM_target, # carbon momentum constraint
    S @ w >= S_target    # ESG score constraint
]

# create the ESG-constrained quadratic optimization problem
prob = cp.Problem(objective, constraints)

# solve the optimization problem and compute portfolio metrics
result_2d = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store target values in the result dictionary
result_2d["CI_target"] = CI_target
result_2d["GI_target"] = GI_target
result_2d["CM_target"] = CM_target
result_2d["S_target"] = S_target

# compute effective variation of carbon intensity relative to the benchmark
result_2d["effective_CI_variation"] = (result_2d["CI_portfolio"] - result_2d["CI_benchmark"]) / result_2d["CI_benchmark"]

# compute effective variation of green intensity relative to the benchmark
result_2d["effective_GI_variation"] = (result_2d["GI_portfolio"] - result_2d["GI_benchmark"]) / result_2d["GI_benchmark"]

# compute effective variation of carbon momentum relative to the benchmark
result_2d["effective_CM_variation"] = (result_2d["CM_portfolio"] - result_2d["CM_benchmark"]) / result_2d["CM_benchmark"]

# compute effective variation of ESG score relative to the benchmark
result_2d["effective_S_variation"] = (result_2d["S_portfolio"] - result_2d["S_benchmark"]) / result_2d["S_benchmark"]

In [18]:
w_star = result_2d["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_2d['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_2d['tracking_error_var']:.2%}")

print(f"\nCI(b): {result_2d['CI_benchmark']:.4f}")
print(f"CI(w*): {result_2d['CI_portfolio']:.4f}")
print(f"CI target: {result_2d['CI_target']:.4f}")
print(f"Carbon Intensity Variation: {result_2d['effective_CI_variation']:.2%}")

print(f"\nCM(b): {result_2d['CM_benchmark']:.2%}")
print(f"CM(w*): {result_2d['CM_portfolio']:.2%}")
print(f"CM target: {result_2d['CM_target']:.2%}")
print(f"Carbon Momentum Variation: {result_2d['effective_CM_variation']:.2%}")

print(f"\nGI(b): {result_2d['GI_benchmark']:.2%}")
print(f"GI(w*): {result_2d['GI_portfolio']:.2%}")
print(f"GI target: {result_2d['GI_target']:.2%}")
print(f"Green Intensity Variation: {result_2d['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_2d['S_benchmark']:.4f}")
print(f"S(w*): {result_2d['S_portfolio']:.4f}")
print(f"S target: {result_2d['S_target']:.4f}")
print(f"ESG Score Variation: {result_2d['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_2d[f'w({sec})']:.2%} | b = {result_2d[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 2.65%
Stock 2: 37.56%
Stock 3: 6.65%
Stock 4: 0.00%
Stock 5: 8.26%
Stock 6: 27.40%
Stock 7: 16.84%
Stock 8: 0.64%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 4.41%
Tracking error var: 0.19%

CI(b): 239.2000
CI(w*): 167.4400
CI target: 167.4400
Carbon Intensity Variation: -30.00%

CM(b): -3.10%
CM(w*): -4.65%
CM target: -4.65%
Carbon Momentum Variation: 50.00%

GI(b): 25.98%
GI(w*): 41.76%
GI target: 38.97%
Green Intensity Variation: 60.75%

S(b): 0.1850
S(w*): 0.6850
S target: 0.6850
ESG Score Variation: 270.27%

--- Sector allocation ---
Sector1: w* = 9.94% | b = 32.50%
Sector2: w* = 90.06% | b = 67.50%


#### (e) Compute the tracking error volatility, the carbon intensity, the carbon momentum, the green intensity, the ESG score, and the sector allocation for the optimized portfolios found in 2.(a), 2.(b), 2.(c), and 2.(d)

In [19]:
# create a table comparing portfolio weights for the benchmark and ESG portfolios
weights_table = pd.DataFrame({
    "Portfolio": ["Benchmark", "2(a)", "2(b)", "2(c)", "2(d)"],

    # store weights of each asset for benchmark and optimized portfolios
    "Asset1": [b[0], result_2a["w_star"][0], result_2b["w_star"][0], result_2c["w_star"][0], result_2d["w_star"][0]],
    "Asset2": [b[1], result_2a["w_star"][1], result_2b["w_star"][1], result_2c["w_star"][1], result_2d["w_star"][1]],
    "Asset3": [b[2], result_2a["w_star"][2], result_2b["w_star"][2], result_2c["w_star"][2], result_2d["w_star"][2]],
    "Asset4": [b[3], result_2a["w_star"][3], result_2b["w_star"][3], result_2c["w_star"][3], result_2d["w_star"][3]],
    "Asset5": [b[4], result_2a["w_star"][4], result_2b["w_star"][4], result_2c["w_star"][4], result_2d["w_star"][4]],
    "Asset6": [b[5], result_2a["w_star"][5], result_2b["w_star"][5], result_2c["w_star"][5], result_2d["w_star"][5]],
    "Asset7": [b[6], result_2a["w_star"][6], result_2b["w_star"][6], result_2c["w_star"][6], result_2d["w_star"][6]],
    "Asset8": [b[7], result_2a["w_star"][7], result_2b["w_star"][7], result_2c["w_star"][7], result_2d["w_star"][7]]
})

display(
    weights_table.style.format({
        col: "{:.2%}" for col in weights_table.columns if col != "Portfolio"
    }).hide(axis="index")
)


summary_table = pd.DataFrame({
    "Portfolio": ["2(a)", "2(b)", "2(c)", "2(d)"],

    "TE vol": [
        result_2a["tracking_error_vol"],
        result_2b["tracking_error_vol"],
        result_2c["tracking_error_vol"],
        result_2d["tracking_error_vol"]
    ],

    "CI(w)": [
        result_2a["CI_portfolio"],
        result_2b["CI_portfolio"],
        result_2c["CI_portfolio"],
        result_2d["CI_portfolio"]
    ],

    "CM(w)": [
        result_2a["CM_portfolio"],
        result_2b["CM_portfolio"],
        result_2c["CM_portfolio"],
        result_2d["CM_portfolio"]
    ],

    "GI(w)": [
        result_2a["GI_portfolio"],
        result_2b["GI_portfolio"],
        result_2c["GI_portfolio"],
        result_2d["GI_portfolio"]
    ],

    "S(w)": [
        result_2a["S_portfolio"],
        result_2b["S_portfolio"],
        result_2c["S_portfolio"],
        result_2d["S_portfolio"]
    ],

    "Sector1": [
        result_2a["w(Sector1)"],
        result_2b["w(Sector1)"],
        result_2c["w(Sector1)"],
        result_2d["w(Sector1)"]
    ],

    "Sector2": [
        result_2a["w(Sector2)"],
        result_2b["w(Sector2)"],
        result_2c["w(Sector2)"],
        result_2d["w(Sector2)"]
    ]
})

display(
    summary_table.style.format({
        "TE vol": "{:.2%}",
        "CI(w)": "{:.4f}",
        "CM(w)": "{:.2%}",
        "GI(w)": "{:.2%}",
        "S(w)": "{:.4f}",
        "Sector1": "{:.2%}",
        "Sector2": "{:.2%}"
    })
)

Portfolio,Asset1,Asset2,Asset3,Asset4,Asset5,Asset6,Asset7,Asset8
Benchmark,9.50%,15.50%,5.50%,8.50%,10.00%,25.00%,17.00%,9.00%
2(a),16.62%,16.28%,6.45%,0.00%,9.21%,25.77%,19.98%,5.69%
2(b),6.38%,29.82%,3.20%,0.00%,8.75%,25.72%,21.90%,4.23%
2(c),7.63%,29.08%,4.18%,0.00%,6.74%,25.95%,22.82%,3.60%
2(d),2.65%,37.56%,6.65%,0.00%,8.26%,27.40%,16.84%,0.64%


,Portfolio,TE vol,CI(w),CM(w),GI(w),S(w),Sector1,Sector2
0,2(a),1.33%,167.4400,-3.73%,28.22%,-0.1209,28.76%,71.24%
1,2(b),3.21%,167.4400,-4.38%,38.97%,0.3614,13.82%,86.18%
2,2(c),3.29%,167.4400,-4.65%,38.97%,0.3440,15.41%,84.59%
3,2(d),4.41%,167.4400,-4.65%,41.76%,0.6850,9.94%,90.06%


#### (f) Compute the implied expected return and the alpha of stock i priced in by each optimized portfolio w:

In [20]:
# define a function to compute implied expected returns and alphas for a portfolio
def compute_implied_and_alpha(Sigma, mu, r, mu_implied, w_star):
    # compute portfolio expected return
    mu_w = float(mu @ w_star)

    # compute portfolio volatility
    sigma_w = float(np.sqrt(w_star @ Sigma @ w_star))

    # compute portfolio Sharpe ratio
    SR_w = (mu_w - r) / sigma_w

    # compute implied expected returns associated with portfolio weights
    mu_tilde_w = r + SR_w * (Sigma @ w_star) / sigma_w

    # compute alpha relative to benchmark-implied expected returns
    alpha_w_b = mu_implied_i - mu_tilde_w

    return mu_tilde_w, alpha_w_b


# compute implied returns and alphas for all optimized portfolios
mu_tilde_2a, alpha_2a = compute_implied_and_alpha(Sigma, mu, r, mu_implied_i, result_2a["w_star"])
mu_tilde_2b, alpha_2b = compute_implied_and_alpha(Sigma, mu, r, mu_implied_i, result_2b["w_star"])
mu_tilde_2c, alpha_2c = compute_implied_and_alpha(Sigma, mu, r, mu_implied_i, result_2c["w_star"])
mu_tilde_2d, alpha_2d = compute_implied_and_alpha(Sigma, mu, r, mu_implied_i, result_2d["w_star"])


# create a summary table with implied returns and alphas for each stock
df_f = pd.DataFrame({
    "Stock": np.arange(1, len(mu_implied_i) + 1),

    "mu_tilde^(2a)": mu_tilde_2a,
    "alpha(2a|b)": alpha_2a,

    "mu_tilde^(2b)": mu_tilde_2b,
    "alpha(2b|b)": alpha_2b,

    "mu_tilde^(2c)": mu_tilde_2c,
    "alpha(2c|b)": alpha_2c,

    "mu_tilde^(2d)": mu_tilde_2d,
    "alpha(2d|b)": alpha_2d
})

print("\n--- Implied returns and alphas for all optimized portfolios ---\n")

# display the formatted table
display(
    df_f.style
    .format({
        "mu_tilde^(2a)": "{:.2%}",
        "alpha(2a|b)": "{:.2%}",
        "mu_tilde^(2b)": "{:.2%}",
        "alpha(2b|b)": "{:.2%}",
        "mu_tilde^(2c)": "{:.2%}",
        "alpha(2c|b)": "{:.2%}",
        "mu_tilde^(2d)": "{:.2%}",
        "alpha(2d|b)": "{:.2%}"
    })
    .hide(axis="index")
)


--- Implied returns and alphas for all optimized portfolios ---



Stock,mu_tilde^(2a),alpha(2a|b),mu_tilde^(2b),alpha(2b|b),mu_tilde^(2c),alpha(2c|b),mu_tilde^(2d),alpha(2d|b)
1,3.02%,-0.12%,2.93%,-0.03%,2.94%,-0.04%,2.86%,0.05%
2,3.10%,-0.07%,3.35%,-0.32%,3.36%,-0.33%,3.36%,-0.34%
3,3.78%,-0.00%,3.70%,0.08%,3.71%,0.07%,3.67%,0.11%
4,3.76%,-0.06%,3.79%,-0.09%,3.83%,-0.13%,3.81%,-0.11%
5,6.87%,-0.19%,6.78%,-0.10%,6.67%,0.01%,6.66%,0.02%
6,15.28%,-0.30%,15.21%,-0.22%,15.30%,-0.31%,15.16%,-0.17%
7,6.66%,-0.15%,6.84%,-0.33%,6.86%,-0.36%,6.61%,-0.11%
8,5.06%,0.05%,4.98%,0.13%,4.98%,0.14%,4.59%,0.53%


### Point 3)

#### (a) Compute the carbon intensity of sectors Sector1 and Sector2.

In [21]:
def sector_metric(weights, metric, sector, sector_name):

    # select the assets that belong to the specified sector
    mask = (sector == sector_name)

    # extract portfolio weights and metric values for those assets
    weights_sector = weights[mask]
    metric_sector = metric[mask]

    # normalize sector weights so that they sum to 1 within the sector
    weights_sector_norm = weights_sector / weights_sector.sum()

    # compute the weighted average metric for the sector
    return float(weights_sector_norm @ metric_sector)


# compute benchmark carbon intensity for Sector1
CI_sector1_b = sector_metric(b, CI, sector, "Sector1")

# compute benchmark carbon intensity for Sector2
CI_sector2_b = sector_metric(b, CI, sector, "Sector2")

In [22]:
print("\n--- Carbon intensity by sector (benchmark) ---\n")
print(f"CI(Sector1): {CI_sector1_b:.4f}")
print(f"CI(Sector2): {CI_sector2_b:.4f}")


--- Carbon intensity by sector (benchmark) ---

CI(Sector1): 459.2308
CI(Sector2): 133.2593


#### (b) What is the optimization problem if we impose to reduce the carbon intensity of Sector1 by 30% and the carbon intensity of Sector2 by 50%? Find the optimized portfolio.

In [23]:
# compute benchmark sector carbon intensities (already defined in 3a)
CI_sector1_b = sector_metric(b, CI, sector, "Sector1")
CI_sector2_b = sector_metric(b, CI, sector, "Sector2")

# define sector-specific carbon intensity targets
CI_sector1_target = 0.70 * CI_sector1_b   # 30% reduction for Sector1
CI_sector2_target = 0.50 * CI_sector2_b   # 50% reduction for Sector2

# number of assets
n = len(b)

# define matrices for the tracking-error optimization
Q = Sigma          
R = Sigma @ b      

# portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to the benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# create boolean masks identifying assets belonging to each sector
mask1 = (sector == "Sector1")
mask2 = (sector == "Sector2")

# fully invested, long-only, and sector-level carbon intensity constraints
constraints = [
    cp.sum(w) == 1,
    w >= 0,

    # sector carbon-intensity constraints
    (CI[mask1] - CI_sector1_target) @ w[mask1] <= 0,
    (CI[mask2] - CI_sector2_target) @ w[mask2] <= 0
]

# create the ESG-constrained quadratic optimization problem
prob = cp.Problem(objective, constraints)

# solve the optimization and compute portfolio metrics
result_3b = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store benchmark and target sector carbon intensities
result_3b["CI_sector1_benchmark"] = CI_sector1_b
result_3b["CI_sector2_benchmark"] = CI_sector2_b
result_3b["CI_sector1_target"] = CI_sector1_target
result_3b["CI_sector2_target"] = CI_sector2_target

# extract optimal portfolio weights
w_star = result_3b["w_star"]

# compute sector carbon intensities for the optimized portfolio
result_3b["CI_sector1_portfolio"] = sector_metric(w_star, CI, sector, "Sector1")
result_3b["CI_sector2_portfolio"] = sector_metric(w_star, CI, sector, "Sector2")

# compute effective sector carbon intensity variation relative to the benchmark
result_3b["effective_CI_sector1_variation"] = (result_3b["CI_sector1_portfolio"] - result_3b["CI_sector1_benchmark"]) / result_3b["CI_sector1_benchmark"]
result_3b["effective_CI_sector2_variation"] = (result_3b["CI_sector2_portfolio"] - result_3b["CI_sector2_benchmark"]) / result_3b["CI_sector2_benchmark"]

# compute variations of other ESG metrics relative to the benchmark
result_3b["effective_GI_variation"] = (result_3b["GI_portfolio"] - result_3b["GI_benchmark"]) / result_3b["GI_benchmark"]
result_3b["effective_CM_variation"] = (result_3b["CM_portfolio"] - result_3b["CM_benchmark"]) / result_3b["CM_benchmark"]
result_3b["effective_S_variation"] = (result_3b["S_portfolio"] - result_3b["S_benchmark"]) / result_3b["S_benchmark"]

In [24]:
w_star = result_3b["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_3b['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_3b['tracking_error_var']:.4f}")

print(f"\nCI(b): {result_3b['CI_benchmark']:.4f}")
print(f"CI(w*): {result_3b['CI_portfolio']:.4f}")

print(f"\nSector1 CI(b): {result_3b['CI_sector1_benchmark']:.4f}")
print(f"Sector1 CI(w*): {result_3b['CI_sector1_portfolio']:.4f}")
print(f"Sector1 target: {result_3b['CI_sector1_target']:.4f}")
print(f"Sector1 Carbon Intensity Variation: {result_3b['effective_CI_sector1_variation']:.2%}")

print(f"\nSector2 CI(b): {result_3b['CI_sector2_benchmark']:.4f}")
print(f"Sector2 CI(w*): {result_3b['CI_sector2_portfolio']:.4f}")
print(f"Sector2 target: {result_3b['CI_sector2_target']:.4f}")
print(f"Sector2 Carbon Intensity Variation: {result_3b['effective_CI_sector2_variation']:.2%}")

print(f"\nCM(b): {result_3b['CM_benchmark']:.2%}")
print(f"CM(w*): {result_3b['CM_portfolio']:.2%}")
print(f"Carbon Momentum Variation: {result_3b['effective_CM_variation']:.2%}")

print(f"\nGI(b): {result_3b['GI_benchmark']:.2%}")
print(f"GI(w*): {result_3b['GI_portfolio']:.2%}")
print(f"Green Intensity Variation: {result_3b['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_3b['S_benchmark']:.4f}")
print(f"S(w*): {result_3b['S_portfolio']:.4f}")
print(f"ESG Score Variation: {result_3b['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_3b[f'w({sec})']:.2%} | b = {result_3b[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 33.52%
Stock 2: 0.00%
Stock 3: 3.96%
Stock 4: 9.21%
Stock 5: 35.67%
Stock 6: 4.43%
Stock 7: 0.00%
Stock 8: 13.21%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 17.33%
Tracking error var: 0.0300

CI(b): 239.2000
CI(w*): 219.2644

Sector1 CI(b): 459.2308
Sector1 CI(w*): 321.4615
Sector1 target: 321.4615
Sector1 Carbon Intensity Variation: -30.00%

Sector2 CI(b): 133.2593
Sector2 CI(w*): 66.6296
Sector2 target: 66.6296
Sector2 Carbon Intensity Variation: -50.00%

CM(b): -3.10%
CM(w*): 1.02%
Carbon Momentum Variation: -132.95%

GI(b): 25.98%
GI(w*): 5.63%
Green Intensity Variation: -78.34%

S(b): 0.1850
S(w*): -0.7396
ESG Score Variation: -499.80%

--- Sector allocation ---
Sector1: w* = 59.90% | b = 32.50%
Sector2: w* = 40.10% | b = 67.50%


#### (c) Formulate the QP problem if we also add the sector neutrality constraint. Find the optimized portfolio.

In [25]:
# create boolean masks for the two sectors
mask1 = (sector == "Sector1")
mask2 = (sector == "Sector2")

# number of assets
n = len(b)

# define matrices for the tracking-error optimization problem
Q = Sigma          
R = Sigma @ b      

# define portfolio weight variable
w = cp.Variable(n)

# objective: minimize tracking error variance relative to the benchmark
objective = cp.Minimize(0.5 * cp.quad_form(w, Q) - R @ w)

# fully invested, long-only, sector carbon-intensity targets, and sector neutrality
constraints = [
    cp.sum(w) == 1,   # portfolio weights must sum to 1
    w >= 0,           # long-only constraint

    # sector carbon-intensity constraints
    (CI[mask1] - CI_sector1_target) @ w[mask1] <= 0,
    (CI[mask2] - CI_sector2_target) @ w[mask2] <= 0,

    # sector neutrality constraints: keep sector weights equal to the benchmark
    cp.sum(w[mask1]) == b[mask1].sum(),
    cp.sum(w[mask2]) == b[mask2].sum()
]

# create the ESG-constrained quadratic optimization problem
prob = cp.Problem(objective, constraints)

# solve the optimization problem and compute portfolio metrics
result_3c = solve_esg_qp(prob, w, Sigma, b, CI, CM, GI, S, sector)

# store benchmark and target sector carbon intensities
result_3c["CI_sector1_benchmark"] = CI_sector1_b
result_3c["CI_sector2_benchmark"] = CI_sector2_b
result_3c["CI_sector1_target"] = CI_sector1_target
result_3c["CI_sector2_target"] = CI_sector2_target

# extract optimal portfolio weights
w_star = result_3c["w_star"]

# compute sector carbon intensities of the optimized portfolio
result_3c["CI_sector1_portfolio"] = sector_metric(w_star, CI, sector, "Sector1")
result_3c["CI_sector2_portfolio"] = sector_metric(w_star, CI, sector, "Sector2")

# compute effective sector carbon intensity variations relative to the benchmark
result_3c["effective_CI_sector1_variation"] = (result_3c["CI_sector1_portfolio"] - result_3c["CI_sector1_benchmark"]) / result_3c["CI_sector1_benchmark"]
result_3c["effective_CI_sector2_variation"] = (result_3c["CI_sector2_portfolio"] - result_3c["CI_sector2_benchmark"]) / result_3c["CI_sector2_benchmark"]

# compute effective variations of other portfolio ESG metrics
result_3c["effective_GI_variation"] = (result_3c["GI_portfolio"] - result_3c["GI_benchmark"]) / result_3c["GI_benchmark"]
result_3c["effective_CM_variation"] = (result_3c["CM_portfolio"] - result_3c["CM_benchmark"]) / result_3c["CM_benchmark"]
result_3c["effective_S_variation"] = (result_3c["S_portfolio"] - result_3c["S_benchmark"]) / result_3c["S_benchmark"]

In [26]:
w_star = result_3c["w_star"]

print("\n--- Optimal Portfolio ---")

print("\nOptimal weights:")
for i, w_i in enumerate(w_star):
    print(f"Stock {i+1}: {w_i*100:.2f}%")

print("\n--- Metrics ---")
print(f"Sum of weights: {w_star.sum():.4f}")
print(f"Tracking error vol: {result_3c['tracking_error_vol']:.2%}")
print(f"Tracking error var: {result_3c['tracking_error_var']:.4f}")

print(f"\nCI(b): {result_3c['CI_benchmark']:.4f}")
print(f"CI(w*): {result_3c['CI_portfolio']:.4f}")

print(f"\nSector1 CI(b): {result_3c['CI_sector1_benchmark']:.4f}")
print(f"Sector1 CI(w*): {result_3c['CI_sector1_portfolio']:.4f}")
print(f"Sector1 target: {result_3c['CI_sector1_target']:.4f}")
print(f"Sector1 Carbon Intensity Variation: {result_3c['effective_CI_sector1_variation']:.2%}")

print(f"\nSector2 CI(b): {result_3c['CI_sector2_benchmark']:.4f}")
print(f"Sector2 CI(w*): {result_3c['CI_sector2_portfolio']:.4f}")
print(f"Sector2 target: {result_3c['CI_sector2_target']:.4f}")
print(f"Sector2 Carbon Intensity Variation: {result_3c['effective_CI_sector2_variation']:.2%}")

print(f"\nCM(b): {result_3c['CM_benchmark']:.2%}")
print(f"CM(w*): {result_3c['CM_portfolio']:.2%}")
print(f"Carbon Momentum Variation: {result_3c['effective_CM_variation']:.2%}")

print(f"\nGI(b): {result_3c['GI_benchmark']:.2%}")
print(f"GI(w*): {result_3c['GI_portfolio']:.2%}")
print(f"Green Intensity Variation: {result_3c['effective_GI_variation']:.2%}")

print(f"\nS(b): {result_3c['S_benchmark']:.4f}")
print(f"S(w*): {result_3c['S_portfolio']:.4f}")
print(f"ESG Score Variation: {result_3c['effective_S_variation']:.2%}")

print("\n--- Sector allocation ---")
for sec in np.unique(sector):
    print(f"{sec}: w* = {result_3c[f'w({sec})']:.2%} | b = {result_3c[f'b({sec})']:.2%}")


--- Optimal Portfolio ---

Optimal weights:
Stock 1: 18.98%
Stock 2: 0.00%
Stock 3: 0.00%
Stock 4: 4.95%
Stock 5: 60.04%
Stock 6: 7.46%
Stock 7: 0.00%
Stock 8: 8.57%

--- Metrics ---
Sum of weights: 1.0000
Tracking error vol: 20.32%
Tracking error var: 0.0413

CI(b): 239.2000
CI(w*): 149.4500

Sector1 CI(b): 459.2308
Sector1 CI(w*): 321.4615
Sector1 target: 321.4615
Sector1 Carbon Intensity Variation: -30.00%

Sector2 CI(b): 133.2593
Sector2 CI(w*): 66.6296
Sector2 target: 66.6296
Sector2 Carbon Intensity Variation: -50.00%

CM(b): -3.10%
CM(w*): 3.63%
Carbon Momentum Variation: -217.05%

GI(b): 25.98%
GI(w*): 3.86%
Green Intensity Variation: -85.12%

S(b): 0.1850
S(w*): -0.8755
ESG Score Variation: -573.27%

--- Sector allocation ---
Sector1: w* = 32.50% | b = 32.50%
Sector2: w* = 67.50% | b = 67.50%


#### (d) Compute the tracking error volatility, the carbon intensity, the carbon momentum, the green intensity, the ESG score, and the sector allocation for the optimized portfolios found in 3.(b), and 3.(c).

In [27]:
summary_3d = pd.DataFrame({
    "Portfolio": ["3(b)", "3(c)"],

    "TE vol": [
        result_3b["tracking_error_vol"],
        result_3c["tracking_error_vol"]
    ],

    "CI(w)": [
        result_3b["CI_portfolio"],
        result_3c["CI_portfolio"]
    ],

    "CM(w)": [
        result_3b["CM_portfolio"],
        result_3c["CM_portfolio"]
    ],

    "GI(w)": [
        result_3b["GI_portfolio"],
        result_3c["GI_portfolio"]
    ],

    "S(w)": [
        result_3b["S_portfolio"],
        result_3c["S_portfolio"]
    ],

    "w(Sector1)": [
        result_3b["w(Sector1)"],
        result_3c["w(Sector1)"]
    ],

    "w(Sector2)": [
        result_3b["w(Sector2)"],
        result_3c["w(Sector2)"]
    ]
})

In [28]:
print("\n--- Summary of optimized portfolios: 3(b) vs 3(c) ---\n")

display(
    summary_3d.style.format({
        "TE vol": "{:.2%}",
        "CI(w)": "{:.4f}",
        "CM(w)": "{:.2%}",
        "GI(w)": "{:.2%}",
        "S(w)": "{:.4f}",
        "w(Sector1)": "{:.2%}",
        "w(Sector2)": "{:.2%}"
    }).hide(axis="index")
)


--- Summary of optimized portfolios: 3(b) vs 3(c) ---



Portfolio,TE vol,CI(w),CM(w),GI(w),S(w),w(Sector1),w(Sector2)
3(b),17.33%,219.2644,1.02%,5.63%,-0.7396,59.90%,40.10%
3(c),20.32%,149.4500,3.63%,3.86%,-0.8755,32.50%,67.50%
